Obtain ID's of records that have zero subjects.  
Then free to experiment with methods to enrich their metadata.

In [1]:
NO_RECORDS = 150

In [2]:
from datetime import datetime, timezone
from json import JSONDecodeError
from logging.config import dictConfig
from typing import Any, Optional
import psycopg
from psycopg import errors as psycopg_errors
from psycopg.rows import dict_row
from config.logging_config import LOGGING_CONFIG
from config.postgres_config import PostgresConfig
from tasks import transform_batch, add_file_metadata
import os
from fastapi import FastAPI, Query, HTTPException
import logging
from pydantic import BaseModel, Field
from utils.queue_utils import HarvestEventQueue, detect_identifier_type

from lxml import etree as ET
# API calls
import requests


# added for these experiments
from utils.enrichment_methods import get_doi, enrich_subjects_from_openaire, merge_subjects_from_new_method

d:\eosc_dc\metadata-warehouse\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
dictConfig(LOGGING_CONFIG)
logger = logging.getLogger(__name__)

In [4]:
user = os.getenv("POSTGRES_USER")
passwd = os.getenv("POSTGRES_PASSWORD")
host = os.getenv("POSTGRES_ADDRESS", "127.0.0.1")


if host == '':
    host = "127.0.0.1"
    user = os.getenv("POSTGRES_ADMIN")

connection_params = {
    "user" : user,
    "password" : passwd,
    "dbname" : "datasetdb",
    "port" : 5432,
    "host" : host
}

In [5]:
query_records = """
            -- Exploring subjects from record raw metadata

-- Every xpath defines a potential XML structure with corresponding subject tag placement.
-- If raw_metadata conforms to ANY of these xpaths, subjects will be concatenated <subject> tags. (removing array_to_string() will return arrays)
-- If raw_metadata doesn't conform to any path (and probably has no subject tags), subjects will be ''


WITH query AS (
    SELECT
        records.*,
        array_to_string(
                xpath('oai:record/oai:metadata/datacite:resource/datacite:subjects/datacite:subject/text()', raw_metadata,
                    '{{datacite, http://datacite.org/schema/kernel-4},
                      {oai, http://www.openarchives.org/OAI/2.0/}}'
                )::text[] ||
				xpath('oai:record/oai:metadata/oaidc:oai_datacite/oaidc:payload/datacite:resource/datacite:subjects/datacite:subject/text()', raw_metadata,
                    '{{datacite, http://datacite.org/schema/kernel-4},
                      {oai, http://www.openarchives.org/OAI/2.0/},
                      {oaidc, http://schema.datacite.org/oai/oai-1.1/}}'
                )::text[] ||
				xpath('oai:record/oai:metadata/oai:resource/datacite:subjects/datacite:subject/text()', raw_metadata,
                    '{{datacite, http://datacite.org/schema/kernel-4},
                      {oai, http://www.openarchives.org/OAI/2.0/}}'
                )::text[] ||
				xpath('oai:record/oai:metadata/oad:oai_datacite/oad:payload/datacite:resource/datacite:subjects/datacite:subject/text()', raw_metadata,
				-- note the 3 in datacite
                    '{{datacite, http://datacite.org/schema/kernel-3},
					  {oai, http://www.openarchives.org/OAI/2.0/},
     				  {oad, http://schema.datacite.org/oai/oai-1.0/}}'
                )::text[] ||
				-- same path as previous but uses kernel-4 namespace
				xpath('oai:record/oai:metadata/oad:oai_datacite/oad:payload/datacite:resource/datacite:subjects/datacite:subject/text()', raw_metadata,
                    '{{datacite, http://datacite.org/schema/kernel-4},
					  {oai, http://www.openarchives.org/OAI/2.0/},
     				  {oad, http://schema.datacite.org/oai/oai-1.0/}}'
                )::text[]
				, ', '
        ) AS subjects
    FROM public.records
)
SELECT 
    id,
    title,
    raw_metadata,
    subjects
FROM query
WHERE query.subjects = ''
LIMIT %s
;
"""

In [6]:
try:
    with psycopg.connect(**connection_params, row_factory=dict_row) as conn:
        cur = conn.cursor()
        
        cur.execute(query_records, (NO_RECORDS,))
        
        results = cur.fetchall()
        print(results)

        record_ids = [str(row["id"]) for row in results]
        record_titles = [str(row["title"]) for row in  results]
        record_raw_metadata = [str(row["raw_metadata"]) for row in  results]


except Exception as e:
    logger.exception("Failed to fetch harvest runs")
    raise HTTPException(status_code=500, detail=str(e))

[{'id': '3e6069dd-a306-47fc-9dad-72a21b58435f::10.17199/BRIGHTNESS/NMX0002', 'title': 'Sample Data from NMX', 'raw_metadata': '<record xmlns="http://www.openarchives.org/OAI/2.0/" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xmlns:dc="http://purl.org/dc/elements/1.1/">\n  <header>\n    <identifier identifierType="doi">10.17199/BRIGHTNESS/NMX0002</identifier>\n    <setSpec>openaire_data</setSpec>\n    <datestamp>2020-01-01</datestamp>\n  </header>\n  <metadata>\n    <datacite:resource xmlns:rdf="http://www.w3.org/1999/02/22-rdf-syntax-ns#" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xmlns:dcterms="http://purl.org/dc/terms/" xmlns:datacite="http://datacite.org/schema/kernel-4" xmlns="http://namespace.openaire.eu/schema/oaire/" xsi:schemaLocation="http://www.openarchives.org/OAI/2.0/oai_dc/ https://www.openaire.eu/schema/repo-lit/4.0/openaire.xsd">\n      <datacite:titles>\n        <title>Sample Data from NMX</title>\n      </datacite:titles>\n      <datacite:identifier

In [7]:
record_titles

['Sample Data from NMX',
 'Sample Data from multiblade',
 'Sample Data from multigrid',
 'Sample Data from multigrid',
 'Sample Data from multigrid',
 'Sample Data from multigrid',
 'Sample Data from multigrid',
 'Sample Data from NMX',
 'Sample Data from multiblade',
 'Sample Data from multigrid',
 'Sample Data from multiblade',
 'Sample Data from multiblade',
 'Sample Data from multigrid',
 'Sample Data from multigrid',
 'Sample Data from multiblade',
 'Sample Data from multigrid',
 'Sample Data from multiblade',
 'Sample Data from multiblade',
 'Sample Data from multigrid',
 'Sample Data from multiblade',
 'Sample Data from multigrid',
 'Sample Data from NMX',
 'Sample Data from multiblade',
 'Sample Data from NMX',
 'Sample Data from multigrid',
 'Sample Data from multigrid',
 'Sample Data from NMX',
 'Sample Data from multigrid',
 'Sample Data from multiblade',
 'Sample Data from multiblade',
 'Sample Data from NMX',
 'Sample Data from multiblade',
 'Sample Data from multigrid',
 

In [8]:
record_raw_metadata[2]

'<record xmlns="http://www.openarchives.org/OAI/2.0/" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xmlns:dc="http://purl.org/dc/elements/1.1/">\n  <header>\n    <identifier identifierType="doi">10.17199/BRIGHTNESS/MG0011</identifier>\n    <setSpec>openaire_data</setSpec>\n    <datestamp>2020-01-01</datestamp>\n  </header>\n  <metadata>\n    <datacite:resource xmlns:rdf="http://www.w3.org/1999/02/22-rdf-syntax-ns#" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xmlns:dcterms="http://purl.org/dc/terms/" xmlns:datacite="http://datacite.org/schema/kernel-4" xmlns="http://namespace.openaire.eu/schema/oaire/" xsi:schemaLocation="http://www.openarchives.org/OAI/2.0/oai_dc/ https://www.openaire.eu/schema/repo-lit/4.0/openaire.xsd">\n      <datacite:titles>\n        <title>Sample Data from multigrid</title>\n      </datacite:titles>\n      <datacite:identifier identifierType="URL">https://doi.org/10.17199/BRIGHTNESS/MG0011</datacite:identifier>\n      <datacite:descriptions>\n  

In [9]:
get_doi(record_raw_metadata[2])

# DOI check through bigger repositories  (#1)
Take an XML ETree.
Searches for identifier with identifierType DOI.
    - what if there are multiple, and one is under "related works"?
Seaches thru OpenAlex or Openaire - apis?
Data available? N -> end
Y ->
    search for fields: subject, topic, field, discipline, keyword

OpenAIRE API
- they're officialy moving to a Graph API in May 2026
https://api.openaire.eu/graph/swagger-ui/index.html

In [10]:
import logging
logging.getLogger("urllib3").setLevel(logging.WARNING)

In [11]:
doi = get_doi(record_raw_metadata[0]) 

if not doi:
    print("NO DOI")
else:
    print(enrich_subjects_from_openaire(doi))

NO DOI


In [12]:
# get only records that have a DOI
ix_has_doi = []

for i in range(0, len(record_raw_metadata)):
    rec = get_doi(record_raw_metadata[i])
    if rec:
        ix_has_doi.append(i)

print(ix_has_doi)

[95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149]


In [13]:
id_to_enrich = [record_ids[i] for i in ix_has_doi]
doi_to_enrich = [get_doi(record_raw_metadata[i]) for i in ix_has_doi]

print(id_to_enrich)
print(doi_to_enrich)

['5d1a3aed-114c-43da-b80b-a953b9424daa::inv/937816672', '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1925696878', '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1567247573', '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1057230191', '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1327877749', '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1186974303', '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/105493981', '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2376080733', '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1308543762', '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/652448489', '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/778027846', '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2116848937', '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1305997290', '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2009919718', '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2248510057', '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2093751889', '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2313092576', '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/15314071

In [14]:
# limit to smaller section
#id_to_enrich = id_to_enrich[:30]
#doi_to_enrich = doi_to_enrich[:30]

In [15]:
from utils.enrichment_methods import _openaire_batch_doi_resolve, _opeanaire_get_doi_subjects, connect_ids_with_subjects_by_doi, batch_enrich_subjects_from_openaire

api_result = _openaire_batch_doi_resolve(doi_to_enrich)

print(api_result[:3])

[{'authors': [{'fullName': 'TAUPITZ, Matthias', 'name': 'Matthias', 'surname': 'TAUPITZ', 'rank': 1, 'pid': None}, {'fullName': 'HESSE, Bernhard', 'name': 'Bernhard', 'surname': 'HESSE', 'rank': 2, 'pid': None}, {'fullName': 'UCA, Yavuz', 'name': 'Yavuz', 'surname': 'UCA', 'rank': 3, 'pid': None}, {'fullName': 'STIER, Deborah', 'name': 'Deborah', 'surname': 'STIER', 'rank': 4, 'pid': None}, {'fullName': 'SEIM, Christian', 'name': 'Christian', 'surname': 'SEIM', 'rank': 5, 'pid': None}], 'openAccessColor': None, 'publiclyFunded': None, 'type': 'dataset', 'language': {'code': 'eng', 'label': 'English'}, 'countries': None, 'subjects': [{'subject': {'scheme': 'keyword', 'value': 'id21'}, 'provenance': None}, {'subject': {'scheme': 'keyword', 'value': 'MD-1163'}, 'provenance': None}, {'subject': {'scheme': 'keyword', 'value': 'Medicine'}, 'provenance': None}], 'mainTitle': 'On the underlying mechanisms of chemical binding and accumulation in heart and brain tissue of Gadolinium of Gadoliniu

In [16]:
doi_subjects = _opeanaire_get_doi_subjects(api_result)
doi_subjects

[{'doi': '10.15151/esrf-es-105493981',
  'subjects': ['id21', 'MD-1163', 'Medicine']},
 {'doi': '10.15151/esrf-es-1186974303',
  'subjects': ['Chemistry', 'BM01', 'CH-6536']},
 {'doi': '10.15151/esrf-es-1932936356',
  'subjects': ['LS-3370',
   'x-ray probe',
   'BM16',
   'Life Sciences',
   'ID16A',
   'XRAYS, X-ray Radiation Technique']},
 {'doi': '10.15151/esrf-es-2376080733',
  'subjects': ['LS-3370',
   'x-ray probe',
   'BM16',
   'Life Sciences',
   'ID16A',
   'XRAYS, X-ray Radiation Technique']},
 {'doi': '10.15151/esrf-es-1837445867',
  'subjects': ['PaN',
   'LS-3391',
   'x-ray probe',
   'Particles, Nuclei and Fields',
   'Life Sciences',
   'BM18']},
 {'doi': '10.15151/esrf-es-1925696878',
  'subjects': ['PaN',
   'LS-3391',
   'x-ray probe',
   'Particles, Nuclei and Fields',
   'Life Sciences',
   'BM18']},
 {'doi': '10.15151/esrf-es-652448489',
  'subjects': ['PaN',
   'ID22',
   'Hard Condensed Matter Science',
   'Particles, Nuclei and Fields',
   'HC-4782']},
 {'do

In [17]:
id_dois = []
for ix in range(len(id_to_enrich)):
    id_dois.append(
        {
            'id': id_to_enrich[ix],
            'doi': doi_to_enrich[ix].lower(),
        }
    )
id_dois

[{'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/937816672',
  'doi': '10.15151/esrf-es-937816672'},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1925696878',
  'doi': '10.15151/esrf-es-1925696878'},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1567247573',
  'doi': '10.15151/esrf-es-1567247573'},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1057230191',
  'doi': '10.15151/esrf-es-1057230191'},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1327877749',
  'doi': '10.15151/esrf-es-1327877749'},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1186974303',
  'doi': '10.15151/esrf-es-1186974303'},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/105493981',
  'doi': '10.15151/esrf-es-105493981'},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2376080733',
  'doi': '10.15151/esrf-es-2376080733'},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1308543762',
  'doi': '10.15151/esrf-es-1308543762'},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/65244848

In [18]:
enriched_list = connect_ids_with_subjects_by_doi(id_dois, doi_subjects)
enriched_list

[{'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/937816672',
  'doi': '10.15151/esrf-es-937816672',
  'enriched_subjects': {'BM31',
   'CH-6465',
   'Chemistry',
   'PaN',
   'Particles, Nuclei and Fields'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1925696878',
  'doi': '10.15151/esrf-es-1925696878',
  'enriched_subjects': {'BM18',
   'LS-3391',
   'Life Sciences',
   'PaN',
   'Particles, Nuclei and Fields',
   'x-ray probe'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1567247573',
  'doi': '10.15151/esrf-es-1567247573',
  'enriched_subjects': {'BM18',
   'LS-3320',
   'Life Sciences',
   'PaN',
   'Particles, Nuclei and Fields'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1057230191',
  'doi': '10.15151/esrf-es-1057230191',
  'enriched_subjects': {'BM07',
   'MX-2447',
   'Macromolecular Crystallography',
   'PaN',
   'Particles, Nuclei and Fields'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1327877749',
  'doi': '10.15151/esrf-es-1327877749',

In [19]:
# Only succesfully enriched IDs and subjects.
# Could be useful for comparing how many ids we sent to be enriched vs
# how many actually got new subjects
[el for el in enriched_list if el['enriched_subjects'] is not None]

[{'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/937816672',
  'doi': '10.15151/esrf-es-937816672',
  'enriched_subjects': {'BM31',
   'CH-6465',
   'Chemistry',
   'PaN',
   'Particles, Nuclei and Fields'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1925696878',
  'doi': '10.15151/esrf-es-1925696878',
  'enriched_subjects': {'BM18',
   'LS-3391',
   'Life Sciences',
   'PaN',
   'Particles, Nuclei and Fields',
   'x-ray probe'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1567247573',
  'doi': '10.15151/esrf-es-1567247573',
  'enriched_subjects': {'BM18',
   'LS-3320',
   'Life Sciences',
   'PaN',
   'Particles, Nuclei and Fields'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1057230191',
  'doi': '10.15151/esrf-es-1057230191',
  'enriched_subjects': {'BM07',
   'MX-2447',
   'Macromolecular Crystallography',
   'PaN',
   'Particles, Nuclei and Fields'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1327877749',
  'doi': '10.15151/esrf-es-1327877749',

In [20]:
# All-in-one function

batch_enrich_subjects_from_openaire(id_dois)

[{'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/937816672',
  'doi': '10.15151/esrf-es-937816672',
  'enriched_subjects': {'BM31',
   'CH-6465',
   'Chemistry',
   'PaN',
   'Particles, Nuclei and Fields'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1925696878',
  'doi': '10.15151/esrf-es-1925696878',
  'enriched_subjects': {'BM18',
   'LS-3391',
   'Life Sciences',
   'PaN',
   'Particles, Nuclei and Fields',
   'x-ray probe'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1567247573',
  'doi': '10.15151/esrf-es-1567247573',
  'enriched_subjects': {'BM18',
   'LS-3320',
   'Life Sciences',
   'PaN',
   'Particles, Nuclei and Fields'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1057230191',
  'doi': '10.15151/esrf-es-1057230191',
  'enriched_subjects': {'BM07',
   'MX-2447',
   'Macromolecular Crystallography',
   'PaN',
   'Particles, Nuclei and Fields'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1327877749',
  'doi': '10.15151/esrf-es-1327877749',

In [21]:
# Comparison: Number of initial records vs number of records that have gained subjects via this method
enriched = batch_enrich_subjects_from_openaire(id_dois)
print(len([el['doi'] for el in enriched]))
print(len([el['doi'] for el in enriched if el['enriched_subjects'] is not None]))

54
54


In [22]:
# Taking a look at the metadata initially obtained from DB, subset of few records
full_info_enriched = [el for el in enriched if el['enriched_subjects'] is not None]

first_enriched = full_info_enriched[:10]

In [23]:
# indexes that have been enriched
ix_original = [record_ids.index(rec['id']) for rec in first_enriched]
ix_original

[95, 96, 97, 98, 99, 100, 101, 102, 103, 104]

In [24]:
first_enriched

[{'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/937816672',
  'doi': '10.15151/esrf-es-937816672',
  'enriched_subjects': {'BM31',
   'CH-6465',
   'Chemistry',
   'PaN',
   'Particles, Nuclei and Fields'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1925696878',
  'doi': '10.15151/esrf-es-1925696878',
  'enriched_subjects': {'BM18',
   'LS-3391',
   'Life Sciences',
   'PaN',
   'Particles, Nuclei and Fields',
   'x-ray probe'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1567247573',
  'doi': '10.15151/esrf-es-1567247573',
  'enriched_subjects': {'BM18',
   'LS-3320',
   'Life Sciences',
   'PaN',
   'Particles, Nuclei and Fields'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1057230191',
  'doi': '10.15151/esrf-es-1057230191',
  'enriched_subjects': {'BM07',
   'MX-2447',
   'Macromolecular Crystallography',
   'PaN',
   'Particles, Nuclei and Fields'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1327877749',
  'doi': '10.15151/esrf-es-1327877749',

In [25]:
[record_raw_metadata[i] for i in ix_original]

['<record xmlns="http://www.openarchives.org/OAI/2.0/" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance">\n  <header>\n    <identifier>oai:icatplus.esrf.fr:inv/937816672</identifier>\n    <datestamp>2022-11-07T07:00:23.673Z</datestamp>\n  </header>\n  <metadata>\n    <oai_datacite xmlns="http://schema.datacite.org/oai/oai-1.1/" xsi:schemaLocation="http://schema.datacite.org/oai/oai-1.1/ http://schema.datacite.org/oai/oai-1.1/oai.xsd">\n      <schemaVersion>4.2</schemaVersion>\n      <datacentreSymbol>EI</datacentreSymbol>\n      <payload>\n        <resource xmlns="http://schema.datacite.org/meta/kernel-4.2/" xsi:schemaLocation="http://schema.datacite.org/meta/kernel-4.2/ http://schema.datacite.org/meta/kernel-4.2/metadata.xsd"><identifier identifierType="DOI">10.15151/ESRF-ES-937816672</identifier><creators><creator><creatorName nameType="Personal">Wouter VAN BEEK</creatorName><givenName>Wouter</givenName><familyName>Van Beek</familyName></creator><creator><creatorName nameType="Pe

## OpenAlex API
https://developers.openalex.org/
https://developers.openalex.org/guides/authentication

Get multiple works by DOI

use | between DOIs, up to 100 DOIs at once!!
check out how the results look then.

https://api.openalex.org/works?filter=doi:https://doi.org/10.1234/a|https://doi.org/10.1234/b|https://doi.org/10.1234/c&per_page=100



Use the select parameter to return only specific fields:
https://api.openalex.org/works?select=id,doi,display_name

In [26]:
OpenAlex_API_key = os.getenv("OPENALEX_API_KEY")

In [27]:
OPENALEX_API_URL = "https://api.openalex.org/works/"

doi = '10.15151/esrf-es-1473312981'

# We have to append the ID directly in the path
query_url = OPENALEX_API_URL + "https://doi.org/" + doi

response = requests.get(
    query_url,
    params={"api_key": OpenAlex_API_key}, 
    timeout=30
)

data = response.json()


In [28]:
oa_DOIS = [
    '10.15151/esrf-es-1001518064',
    '10.15151/esrf-es-2323561777',
    '10.15151/esrf-es-2323561778',
    '10.15151/esrf-es-2323561877',
    'https://doi.org/10.5286/ISIS.E.100051344',
    '10.34894/AUUSS2',
    '10.17026/DANS-28H-C53V',
    '10.34894/VLSN1K',
    '10.15151/ESRF-ES-1030397492',
    '10.17026/AR/DMHU53',
    '10.17026/AR/GHYQTO',
    '10.17026/DANS-ZU2-HF3N',
    '10.17026/DANS-ZTM-7SA5',
    '10.17026/DANS-ZUK-HN2Z',
    'https://doi.org/10.1016/j.paid.2021.110888',
    '10.23662/fors-ds-686-1',
    '10.1056/NEJMoa2034577',
    '10.1056/NEJMoa2021436',
    '10.1056/NEJMoa1511939',
    '10.1056/NEJMoa1800389'
]

the API result object can have a result for a given DOI, but that result doesn't contain any keywords = doesn't contribute to enrichment
Hence the len(api_result) can be larger than len(enriched_dois)

Additionaly if we set a high keyword score threshold, a certain result MIGHT have keywords but none are relevant enough - that will also not contribute to enrichment

In [29]:
from utils.enrichment_methods import batch_enrich_subjects_from_openalex

enriched_results = batch_enrich_subjects_from_openalex(
    id_dois,
    OpenAlex_API_key,
    score_treshold = 0.6
)

enriched_results = [el for el in enriched_results if el['enriched_subjects']]
print(len(enriched_results))
enriched_results

6


[{'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2376080733',
  'doi': '10.15151/esrf-es-2376080733',
  'enriched_subjects': {'Context (archaeology)',
   'Multicellular organism',
   'Sample preparation',
   'Workflow'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2248510057',
  'doi': '10.15151/esrf-es-2248510057',
  'enriched_subjects': {'Redox'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2313092576',
  'doi': '10.15151/esrf-es-2313092576',
  'enriched_subjects': {'Cathodoluminescence',
   'Dark field microscopy',
   'Diffraction'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2356598719',
  'doi': '10.15151/esrf-es-2356598719',
  'enriched_subjects': {'Amorphous metal',
   'Amorphous solid',
   'Differential scanning calorimetry',
   'Supercooling'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2312906364',
  'doi': '10.15151/esrf-es-2312906364',
  'enriched_subjects': {'Digitization'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2348957234'

In [30]:
# Showcase  - manually input DOIs

test_dois = [
    {'id': "testid1",
     'doi': '10.1016/j.paid.2021.110888'},
    {'id': "testid2",
     'doi': '10.23662/fors-ds-686-1'}
]

enriched_results = batch_enrich_subjects_from_openalex(
    test_dois,
    OpenAlex_API_key,
    score_treshold = 0.7
)

enriched_results

[{'id': 'testid1',
  'doi': '10.1016/j.paid.2021.110888',
  'enriched_subjects': {'Conscientiousness',
   'Flow Experience in Various Fields',
   'Mind wandering and attention',
   'Psychology',
   'Sport Psychology and Performance'}},
 {'id': 'testid2',
  'doi': '10.23662/fors-ds-686-1',
  'enriched_subjects': set()}]

In [31]:
# Compare openalex with openaire:
oaire_results = batch_enrich_subjects_from_openaire(test_dois)
oaire_results

[{'id': 'testid1',
  'doi': '10.1016/j.paid.2021.110888',
  'enriched_subjects': {'05 social sciences',
   '0501 psychology and cognitive sciences',
   '10. No inequality'}},
 {'id': 'testid2',
  'doi': '10.23662/fors-ds-686-1',
  'enriched_subjects': {'10. No inequality',
   '3. Good health',
   '4. Education',
   '8. Economic growth'}}]

In [32]:
merge_subjects_from_new_method(enriched_results, oaire_results)

[{'id': 'testid1',
  'doi': '10.1016/j.paid.2021.110888',
  'enriched_subjects': {'05 social sciences',
   '0501 psychology and cognitive sciences',
   '10. No inequality',
   'Conscientiousness',
   'Flow Experience in Various Fields',
   'Mind wandering and attention',
   'Psychology',
   'Sport Psychology and Performance'}},
 {'id': 'testid2',
  'doi': '10.23662/fors-ds-686-1',
  'enriched_subjects': {'10. No inequality',
   '3. Good health',
   '4. Education',
   '8. Economic growth'}}]

# DOI resolver method  (#2)
Take an XML ETree.
Searches for identifier with identifierType DOI.
    - what if there are multiple, and one is under "related works"?
Talks to an API such as DataCite or SignPosting.
Data available? N -> end
Y ->
    search for fields: subject, topic, field, discipline, keyword

# Related DOIs  
Extract related identifiers that are also DOIs. If they are in the database, use their subjects. If not, use DOI-resolving methods on them.

Note:
a DOI must start with 10. prefix. I've seen some related DOIs that start with 0.  , if I add a 1 to the start it resolves properly.

## SQL query to see records with related DOIs   
```
--- get records with related identifiers of any tipe
WITH query AS (
    SELECT
        raw_metadata,
		doi as record_doi,
        xpath($$(//*[local-name()='relatedIdentifier'][@relatedIdentifierType='DOI'])/text()$$, raw_metadata)::text[] AS related
    FROM public.records
)
SELECT *
FROM query
WHERE array_length(related, 1) > 0
LIMIT 1000;
```

47k records have SOME relatedIdentifiers - lots of them ISSN or proprietary ids

In [44]:
# Let's try out some records with related DOIs, for testing we won't limit ourselves to records with no raw_subjects.

query_related_dois = """
WITH query AS (
    SELECT
        id,
        raw_metadata,
		doi as record_doi,
        ARRAY(
            SELECT jsonb_array_elements(datacite_json->'subjects')->>'subject'
        ) AS subjects,
        xpath($$(//*[local-name()='relatedIdentifier'][@relatedIdentifierType='DOI'])/text()$$, raw_metadata)::text[] AS related
    FROM public.records
)
SELECT *
FROM query
WHERE array_length(related, 1) > 0
LIMIT 100;
"""

In [45]:
try:
    with psycopg.connect(**connection_params, row_factory=dict_row) as conn:
        cur = conn.cursor()
        
        cur.execute(query_related_dois)
        
        results = cur.fetchall()
        print(results)

        record_ids = [str(row["id"]) for row in results]
        record_subjects = [row["subjects"] for row in results]
        record_raw_metadata = [str(row["raw_metadata"]) for row in  results]
        record_related_dois = [row["related"] for row in results]


except Exception as e:
    logger.exception("Failed to fetch harvest runs")
    raise HTTPException(status_code=500, detail=str(e))

[{'id': 'a64cfb20-e461-4ec0-a069-a00c1fde6c64::3593', 'raw_metadata': '<record xmlns="http://www.openarchives.org/OAI/2.0/" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance">\n  <header>\n    <identifier>oai:rodare.hzdr.de:3593</identifier>\n    <datestamp>2025-03-14T14:32:07Z</datestamp>\n    <setSpec>openaire_data</setSpec>\n    <setSpec>user-rodare</setSpec>\n  </header>\n  <metadata>\n    <oai_datacite xmlns="http://schema.datacite.org/oai/oai-1.0/" xsi:schemaLocation="http://schema.datacite.org/oai/oai-1.0/ oai_datacite.xsd">\n      <isReferenceQuality>true</isReferenceQuality>\n      <schemaVersion>4.1</schemaVersion>\n      <datacentreSymbol>CERN.ZENODO</datacentreSymbol>\n      <payload>\n        <resource xmlns="http://datacite.org/schema/kernel-4" xsi:schemaLocation="http://datacite.org/schema/kernel-4 http://schema.datacite.org/meta/kernel-4.1/metadata.xsd">\n          <identifier identifierType="DOI">10.14278/rodare.3593</identifier>\n          <creators>\n            <

In [46]:
results

[{'id': 'a64cfb20-e461-4ec0-a069-a00c1fde6c64::3593',
  'raw_metadata': '<record xmlns="http://www.openarchives.org/OAI/2.0/" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance">\n  <header>\n    <identifier>oai:rodare.hzdr.de:3593</identifier>\n    <datestamp>2025-03-14T14:32:07Z</datestamp>\n    <setSpec>openaire_data</setSpec>\n    <setSpec>user-rodare</setSpec>\n  </header>\n  <metadata>\n    <oai_datacite xmlns="http://schema.datacite.org/oai/oai-1.0/" xsi:schemaLocation="http://schema.datacite.org/oai/oai-1.0/ oai_datacite.xsd">\n      <isReferenceQuality>true</isReferenceQuality>\n      <schemaVersion>4.1</schemaVersion>\n      <datacentreSymbol>CERN.ZENODO</datacentreSymbol>\n      <payload>\n        <resource xmlns="http://datacite.org/schema/kernel-4" xsi:schemaLocation="http://datacite.org/schema/kernel-4 http://schema.datacite.org/meta/kernel-4.1/metadata.xsd">\n          <identifier identifierType="DOI">10.14278/rodare.3593</identifier>\n          <creators>\n           

In [64]:
# For a specific DOI, search our datasetdb and retrieve subjects if any.
# Maybe a record references/is part of another dataset so the subjects are relevant too.

query_subjects_by_doi = """
WITH query AS (
    SELECT
    id,
        ARRAY(
            SELECT jsonb_array_elements(datacite_json->'subjects')->>'subject'
        ) AS subjects
    FROM public.records
)
SELECT subjects
FROM query
WHERE query.id = ANY(%s)
LIMIT 100;
"""

In [62]:
first_related_doi = record_related_dois[0]
first_related_doi

['10.14278/rodare.3592']

In [65]:
try:
    with psycopg.connect(**connection_params, row_factory=dict_row) as conn:
        cur = conn.cursor()
        
        cur.execute(query_subjects_by_doi, (first_related_doi,))
        
        record_res = cur.fetchall()
except:
    print(f"Query error: {e}")
        

record_res

[]